# UR5e 状态获取和控制

可直接使用 ur5e_demo 环境，本文件涉及操作仅依赖 numpy、ur-rtde、pyDHgripper 库

## 初始化实例（执行一次即可）

In [ ]:
import numpy as np
import rtde_receive
import rtde_control

UR5E_IP = "192.168.31.123" # 目前 UR5e 配置的是静态地址

rtde_r = rtde_receive.RTDEReceiveInterface(UR5E_IP)
rtde_c = rtde_control.RTDEControlInterface(UR5E_IP)

## 状态获取

In [6]:
print(f"joint positions: {rtde_r.getActualQ()}")
print(f"tcp pose: {rtde_r.getActualTCPPose()}")

joint positions: [-2.6177430788623255, -2.009884019891256, 2.4285035769092005, -0.43123896539721684, 1.307411789894104, -1.5708196798907679]
tcp pose: [0.15893604304454031, 0.27508667091080785, 0.28895907053893977, -0.6128067030956368, 1.4704877849570936, 0.6129567820405573]


## 位姿控制

In [ ]:
# 基于关节的位姿控制（阻塞，到达目标位置后才会继续运行后面的代码）
pre_init_joint = [-2.6177242437945765, -2.009850164453024, 2.4285362402545374, -0.4312353891185303, 1.307356834411621, -np.pi/2]
rtde_c.moveJ(pre_init_joint, speed=0.3, acceleration=1.1)

True

In [ ]:
# 基于 tcp 的位姿控制（阻塞）
pre_init_tcp = [
    0.15892605120155803, 0.2750888828108686, 0.2889513384773401,
    -0.6128288676482984, 1.4705028486671405, 0.6130056068510648
]
rtde_c.moveJ_IK(pre_init_tcp, speed=0.3, acceleration=1.1)

In [2]:
# 基于关节的位姿控制（非阻塞，在向目标位置移动的同时就会继续执行后面的代码，简单理解为机械臂会始终向最新发送的目标位置移动）

import time

point1 = [
    -2.7177242437945765, -2.009850164453024, 2.4285362402545374,
    -0.4312353891185303, 1.307356834411621, -np.pi / 2
]
point2 = [
    -2.6177242437945765, -2.009850164453024, 2.4285362402545374,
    -0.4312353891185303, 1.307356834411621, -np.pi / 2
]

rtde_c.servoJ(point1,
              0.1,
              0.1,
              0.0,
              0.1,
              500.0)
rtde_c.servoJ(point2,
              0.1,
              0.1,
              0.0,
              0.1,
              500.0)
# 机械臂会直接运行到 point2, 非阻塞模式下不会等待到达目标位置，第二条命令会覆盖掉第一条命令
# servoJ 参数列表定义见：https://sdurobotics.gitlab.io/ur_rtde/api/api.html#_CPPv4N7ur_rtde20RTDEControlInterface6servoJERKNSt6vectorIdEEddddd

True

In [3]:
try:
    rtde_c.servoStop()
except:
    pass

try:
    rtde_c.stopScript()
except:
    pass

rtde_r.disconnect()
rtde_c.disconnect()

# DH 夹爪控制

In [ ]:
from pyDHgripper import AG95
import time

# 实例化时会自动校准，会张开闭合一次校准最大最小位置，此过程中不要放东西在夹爪中
gripper = AG95(port='/dev/ttyUSB0') # DH 夹爪直接连接USB到主机，如果夹爪不在 ttyUSB0 上，请依次尝试后续 USB 设备1/2/3...

In [ ]:
gripper.set_pos(0) # close

In [ ]:
gripper.set_pos(1000) # full open

In [ ]:
gripper.set_force(100) # 20~100

In [ ]:
gripper.set_pos(400)